In [ ]:
import os
import logging
import sys
from pathlib import Path
from dotenv import load_dotenv
import tempfile
from typing import Any


TMPDIR = tempfile.mkdtemp()
ROOT_DIR = Path.cwd()

SRC_DIR = ROOT_DIR / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Added to sys.path: {SRC_DIR}")
load_dotenv()  # reads .env file from the current directory

# --- logging setup ---

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# optionally raise the level for noisy third-party loggers
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("great_expectations._docs_decorators").setLevel(logging.WARNING)

logger = logging.getLogger("notebook")
logger.info("logging is configured")

In [ ]:
import pandas as pd
from huggingface_hub import HfApi
from datasets import load_dataset, Dataset, concatenate_datasets
from tgedr_dataops_abs.processor import Processor, ProcessorException
from tgedr_dataops.store.hf_dataset_file_arrow_based import HuggingFaceDatasetFileArrowBasedStore

In [ ]:
ds_bronze_prefix = "jtviegas/faers_bronze_"
ds_silver_prefix = "jtviegas/faers_silver_"

ds_term_entity_url = "jtviegas/faers_silver_term_entity"
ds_entity_atc_url = "jtviegas/faers_silver_entity_atc"
ds_bronze_drug_url = "jtviegas/faers_bronze_drug"
ds_gold_drug_url = "jtviegas/faers_gold_drug"

In [ ]:
o: HuggingFaceDatasetFileArrowBasedStore = HuggingFaceDatasetFileArrowBasedStore()

ds_demo = o.get(key="jtviegas/faers_bronze_demo", streaming=True, columns=["primaryid", "caseid", "caseversion"]).train

In [ ]:
def _version_key(version: Any) -> tuple[int, int]:
    return (0, -1) if version is None else (1, int(version))


latest_by_case: dict[Any, dict[str, Any]] = {}
ambiguous_latest_cases: set[Any] = set()
selected_primaryids: set[int] = set()

for row in ds_demo:
    primaryid = row["primaryid"]
    caseid = row["caseid"]
    version = row["caseversion"]
    if primaryid is None:
        continue
    if caseid is None:
        selected_primaryids.add(primaryid)
        continue

    previous_version = latest_by_case.get(caseid)
    if previous_version is None or _version_key(version) > _version_key(previous_version["caseversion"]):
        latest_by_case[caseid] = row
    elif _version_key(version) == _version_key(previous_version["caseversion"]):
        # ambiguous_latest_cases.add(caseid)
        latest_by_case[caseid] = row

selected_primaryids.update(row["primaryid"] for row in latest_by_case.values())

In [ ]:
from tgedr_fdafaers.etl.faers_structured_content import FaersStructuredContent


o = FaersStructuredContent(
    configuration={"bronze_dataset_prefix": ds_bronze_prefix, "silver_dataset_prefix": ds_silver_prefix}
)
o.extract()

In [ ]:
o.transform()

In [ ]:
# dataset = load_dataset(ds_term_entity, split="train", streaming=False)
# o = ContractedHFDatasetFileBasedStore()

In [ ]:
# o = HuggingFaceDatasetFileArrowBasedStore()
from tgedr_fdafaers.etl.silver_layer import SilverLayer


o = SilverLayer(
    configuration={"bronze_dataset_prefix": "jtviegas/faers_bronze_", "silver_dataset_prefix": "jtviegas/faers_silver_"}
)

In [ ]:
o.extract()

In [ ]:
ds0 = Dataset.from_dict(o._data["drug"].select_columns(["primaryid", "drug_seq", "drugname"])[:12])
ds1 = Dataset.from_dict(
    {"primaryid": [4338027, 2, 3], "drug_seq": [1004795909, 2, 3], "drugname": ["drugA", "drugB", "drugC"]}
)
# ds1 = Dataset.from_dict(
#     {"primaryid": [4338027], "drug_seq": [1004795909], "drugname": ["drugA"]}
# )
ds0[:]

In [ ]:
from typing import List


def update(ds0: Dataset, ds1: Dataset, key_fields: List[str]) -> Any:
    def row_key(row: dict) -> tuple:
        return tuple(row[field] for field in key_fields)

    incoming_keys = {row_key(row) for row in ds1.to_list()}
    existing_keys = {row_key(row) for row in ds0.to_list()}
    updating_keys = incoming_keys & existing_keys

    rows_to_update = ds1.filter(
        lambda row: row_key(row) in updating_keys,
        load_from_cache_file=False,
    )
    rows_to_add = ds1.filter(
        lambda row: row_key(row) not in updating_keys,
        load_from_cache_file=False,
    )
    rows_to_keep = ds0.filter(
        lambda row: row_key(row) not in updating_keys,
        load_from_cache_file=False,
    )

    return concatenate_datasets(
        [rows_to_keep, rows_to_update, rows_to_add]
    )  # if rows_to_add.num_rows else concatenate_datasets([rows_to_keep, rows_to_update])

In [ ]:
update(ds0, ds1, key_fields=["primaryid", "drug_seq"])[:]

In [ ]:
o.transform()

In [ ]:
data: dict[str, Dataset] = {}
data["term_entity"] = o.get(key=ds_term_entity_url, streaming=True).train
data["entity_atc"] = o.get(key=ds_entity_atc_url, streaming=False).train

In [ ]:
entity_atc_lookup: dict[str, list[Any]] = {}
for row in data["entity_atc"]:
    entity_atc_lookup.setdefault(row["entity"], []).extend(row["atc_codes"])

term_atc_map: dict[str, list[Any]] = {}
for te in data["term_entity"]:
    term = te["term"]
    entity = te["entity"]
    atc_codes = entity_atc_lookup.get(entity, [])
    if term not in term_atc_map:
        term_atc_map[term] = []
    term_atc_map[term].extend(atc_codes)

rows = [{"term": term, "atc_codes": atc_codes} for term, atc_codes in term_atc_map.items()]

In [ ]:
dataset = Dataset.from_list(rows)
dataset

In [ ]:
Dataset.from_dict(term_atc_map, features=feature)

In [ ]:
ds_term_entity = o.get(ds_term_entity_url).train
term_entity_map: dict[str, list[str]] = {}
for row in ds_term_entity:
    term = row["term"]
    entity = row["entity"]
    term_entity_map[term] = [entity] if term not in term_entity_map else term_entity_map[term] + [entity]
ds_term_entity

In [ ]:
ds_entity_atc = o.get(ds_entity_atc_url).train
entity_atc_map: dict[str, list[str]] = {}
for row in ds_entity_atc:
    entity = row["entity"]
    atc = row["atc_codes"]
    entity_atc_map[entity] = [atc] if entity not in entity_atc_map else entity_atc_map[entity] + [atc]

ds_entity_atc

In [ ]:
ds_bronze_drug = o.get(ds_bronze_drug_url).train
ds_bronze_drug

In [ ]:
from datasets import Dataset

# term_entity_map: dict[str, list[str]] = {}
# entity_atc_map: dict[str, list[str]] = {}
term_atc_map = {}

for term, entities in term_entity_map.items():
    atc_codes = []
    for entity in entities:
        atc_codes.extend(entity_atc_map.get(entity, []))
    term_atc_map[term] = atc_codes


def map_term_to_atc(row):
    term = row["term"]
    new_row = dict(row)
    new_row["atc_codes"] = term_atc_map.get(term, [])
    return new_row


ds = ds_bronze_drug.map(map_term_to_atc)

In [ ]:
ds

In [ ]:
ds[:10]